# Internal Dependencies — SCIP Semantic Index
<br>

Explores SCIP (Semantic Code Intelligence Protocol) module and artifact dependency structure.
SCIP indexes are language-agnostic and support TypeScript, Go, Rust, and other languages.

### References
- [SCIP — Semantic Code Intelligence Protocol](https://github.com/sourcegraph/scip)
- [code-graph-analysis-pipeline — SCIP support](../../SCIP.md)
- [Neo4j Python Driver](https://neo4j.com/docs/api/python-driver/current)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plot
from neo4j import GraphDatabase

In [ ]:
# Please set the environment variable "NEO4J_INITIAL_PASSWORD" in your shell 
# before starting jupyter notebook to provide the password for the user "neo4j".
# It is not recommended to hardcode the password into jupyter notebook for security reasons.

driver = GraphDatabase.driver(uri="bolt://localhost:7687", auth=("neo4j", os.environ.get("NEO4J_INITIAL_PASSWORD")))
driver.verify_connectivity()

In [ ]:
def get_cypher_query_from_file(cypher_file_name: str) -> str:
    with open(cypher_file_name) as file:
        return ' '.join(file.readlines())


def query_cypher_to_data_frame(filename: str, limit: int = -1) -> pd.DataFrame:
    """
    Execute the Cypher query of the given file and returns the result as a DataFrame.

    Args:
        filename: Path to the file containing the Cypher query.
        limit: Optional row limit appended to the query. Default -1 = no limit.
    """
    cypher_query = get_cypher_query_from_file(filename)
    if limit > 0:
        cypher_query = f"{cypher_query}\nLIMIT {limit}"
    records, summary, keys = driver.execute_query(cypher_query)
    return pd.DataFrame([r.values() for r in records], columns=keys)

In [ ]:
#The following cell uses the build-in %html "magic" to override the CSS style for tables to a much smaller size.
#This is especially needed for PDF export of tables with multiple columns.

In [ ]:
%%html
<style>
/* CSS style for smaller dataframe tables. */
.dataframe th {
    font-size: 8px;
}
.dataframe td {
    font-size: 8px;
}
</style>

In [ ]:
# Pandas DataFrame Display Configuration
pd.set_option('display.max_colwidth', 300)

## 1 — SCIP Modules

SCIP modules correspond to source directories. The following tables sort modules by different metrics to highlight central or leaf modules.

The full list is in the CSV report: `reports/internal-dependencies/SCIP_Semantic_Index_Module/List_all_SCIP_modules.csv`

In [ ]:
scip_modules = query_cypher_to_data_frame("../queries/internal-dependencies/List_all_SCIP_modules.cypher")

### Table 1a — Top 30 modules with the highest type count

In [ ]:
scip_modules.sort_values(by=['numberOfTypes', 'moduleName'], ascending=[False, True]).reset_index(drop=True).head(30)

### Table 1b — Top 30 modules with the highest number of incoming dependencies

High incoming = widely imported by other modules (high in-degree).

In [ ]:
scip_modules.sort_values(by=['incomingDependencies', 'moduleName'], ascending=[False, True]).reset_index(drop=True).head(30)

### Table 1c — Top 30 modules with the highest number of outgoing dependencies

High outgoing = broad consumer depending on many other modules (high out-degree).

In [ ]:
scip_modules.sort_values(by=['outgoingDependencies', 'moduleName'], ascending=[False, True]).reset_index(drop=True).head(30)

### Table 1d — Top 30 modules with the lowest type count

In [ ]:
scip_modules.sort_values(by=['numberOfTypes', 'moduleName'], ascending=[True, True]).reset_index(drop=True).head(30)

### Table 1e — Top 30 modules with the lowest number of incoming dependencies

In [ ]:
scip_modules.sort_values(by=['incomingDependencies', 'moduleName'], ascending=[True, True]).reset_index(drop=True).head(30)

### Table 1f — Top 30 modules with the lowest number of outgoing dependencies

In [ ]:
scip_modules.sort_values(by=['outgoingDependencies', 'moduleName'], ascending=[True, True]).reset_index(drop=True).head(30)

### Table 1g — Test modules only

In [ ]:
scip_modules[scip_modules['isTest'] == True].sort_values(by=['numberOfTypes', 'moduleName'], ascending=[False, True]).reset_index(drop=True).head(30)

## 2 — SCIP Artifacts

SCIP artifacts correspond to packages (npm, go module, cargo crate, etc.).

The full list is in the CSV report: `reports/internal-dependencies/SCIP_Semantic_Index_Artifact/List_all_SCIP_artifacts.csv`

In [ ]:
scip_artifacts = query_cypher_to_data_frame("../queries/internal-dependencies/List_all_SCIP_artifacts.cypher")

### Table 2a — Top 30 artifacts with the highest number of incoming dependencies

In [ ]:
scip_artifacts.sort_values(by=['incomingDependencies', 'artifactName'], ascending=[False, True]).reset_index(drop=True).head(30)

### Table 2b — Top 30 artifacts with the highest number of outgoing dependencies

In [ ]:
scip_artifacts.sort_values(by=['outgoingDependencies', 'artifactName'], ascending=[False, True]).reset_index(drop=True).head(30)

### Table 2c — Internal artifacts only (not external libraries)

In [ ]:
scip_artifacts[scip_artifacts['isExternal'] == False].sort_values(by=['incomingDependencies', 'artifactName'], ascending=[False, True]).reset_index(drop=True).head(30)